Feature Extraction

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog, local_binary_pattern
from scipy.stats import kurtosis, skew
from tqdm import tqdm

# Dictionary to map person IDs to unique class labels
person_label_map = {}
current_class = 0

def get_person_id(filename):
    """Extracts person ID from filename (assuming consistent naming)."""
    parts = filename.split('_')  # Adjust this based on your actual filename format
    return parts[2]  # Assuming the 3rd part of the filename is the person ID

def assign_label(filename, is_genuine):
    """Assigns a unique label for each person's genuine and forged class."""
    global current_class
    person_id = get_person_id(filename)
    
    if person_id not in person_label_map:
        person_label_map[person_id] = {"genuine": current_class, "forged": current_class + 1}
        current_class += 2
    
    return person_label_map[person_id]["genuine"] if is_genuine else person_label_map[person_id]["forged"]

def extract_features(image_path):
    """Extracts multiple handcrafted features from an image."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, (256, 128))  # Resize to fixed size
    
    features = []
    features.extend(hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2), 
                         orientations=9, visualize=False, block_norm='L2-Hys'))
    lbp = local_binary_pattern(img, P=24, R=3, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 27), range=(0, 26))
    hist = hist.astype("float") / hist.sum()
    features.extend(hist)
    
    moments = cv2.moments(img)
    hu_moments = cv2.HuMoments(moments).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments))
    features.extend(hu_moments)
    
    orb = cv2.ORB_create()
    keypoints, _ = orb.detectAndCompute(img, None)
    features.append(len(keypoints))
    
    features.extend([np.mean(img), np.std(img), np.median(img), skew(img.flatten()), kurtosis(img.flatten())])
    
    binary = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    thickness = np.sum(binary == 255) / np.count_nonzero(binary)
    density = np.sum(binary == 255) / binary.size
    features.extend([thickness, density])
    
    return features

def process_dataset(input_folder, is_genuine):
    """Processes all images in the dataset, extracting features and assigning labels."""
    data = []
    for file_name in tqdm(os.listdir(input_folder), desc=f"Processing {'Genuine' if is_genuine else 'Forged'} Signatures"):
        image_path = os.path.join(input_folder, file_name)
        features = extract_features(image_path)
        if features:
            label = assign_label(file_name, is_genuine)
            data.append([file_name, label] + features)
    return data

# Paths to augmented images
genuine_path = "../Dataset/Processing/AugmentedDataset/Genuine"
forged_path = "../Dataset/Processing/AugmentedDataset/Forged"

# Extract features
genuine_data = process_dataset(genuine_path, True)
forged_data = process_dataset(forged_path, False)

# Create DataFrame
columns = ["filename", "label"] + [f"feature_{i}" for i in range(len(genuine_data[0]) - 2)]
df = pd.DataFrame(genuine_data + forged_data, columns=columns)
df.to_csv("../Dataset/Features/signature_features_test.csv", index=False)

print("Feature extraction completed with person-specific labels and saved!")


Model Training

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.utils import Sequence, to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
from sklearn.model_selection import train_test_split

# Load features and labels
data = pd.read_csv("../Dataset/Features/signature_features_test.csv")

# Convert labels to integers and determine num_classes correctly
labels = data["label"].astype(int).values  # Ensure labels are integers
num_classes = max(labels) + 1  # Ensure num_classes is correctly assigned

# One-hot encode labels
labels = to_categorical(labels, num_classes=num_classes)

# Extract handcrafted features and construct image paths
handcrafted_features = data.iloc[:, 2:].values
image_paths = [f"../Dataset/Processing/AugmentedDataset/Genuine/{file}" if file.startswith('T_') 
               else f"../Dataset/Processing/AugmentedDataset/Forged/{file}" 
               for file in data["filename"]]

# Split dataset
train_img_paths, val_img_paths, train_handcrafted, val_handcrafted, y_train, y_val = train_test_split(
    image_paths, handcrafted_features, labels, test_size=0.2, random_state=42, stratify=labels.argmax(axis=1))

# Custom Data Generator
class SignatureDataGenerator(Sequence):
    def __init__(self, image_paths, handcrafted_features, labels, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.handcrafted_features = handcrafted_features
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_images = np.array([self.load_image(self.image_paths[i]) for i in batch_indexes])
        batch_handcrafted = np.array([self.handcrafted_features[i] for i in batch_indexes])
        batch_labels = np.array([self.labels[i] for i in batch_indexes])

        return (batch_images, batch_handcrafted.reshape(-1, batch_handcrafted.shape[1], 1)), batch_labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

# Create data generators
train_generator = SignatureDataGenerator(train_img_paths, train_handcrafted, y_train)
val_generator = SignatureDataGenerator(val_img_paths, val_handcrafted, y_val)

# Define model
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)

lstm_input = Input(shape=(train_handcrafted.shape[1], 1))
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)

merged = Concatenate()([cnn_output, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(num_classes, activation='softmax')(dropout_layer)

model = Model(inputs=[inception_base.input, lstm_input], outputs=output_layer)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train model
model.fit(train_generator, validation_data=val_generator, epochs=25)

print("Model training completed successfully!")
